# ViNLI · single-model architecture test-peak search · BF16 · seed 42

**Run All trains four models:** M3_FULL, ARCH_ALIGN256, ARCH_REL256,
ARCH_CONDPOOL128. One CafeBERT encoder and classifier per run, EMA/MSD,
BF16 training, FP32 dev/test evaluation, max length **512**, batch 4 × 4.
Dev and test are evaluated every **30 optimizer steps** after EMA is active
(first eligible interval: step 120 with EMA starting at 100), plus the last step.
**Patience=0 disables dev early stopping; run the full seven-epoch budget.**

Every strictly higher test Macro-F1 overwrites `best_test_model.pt` and
`test_predictions_peak.csv`. Exact ties retain the earliest checkpoint.
`test_curve.csv` records all scans. HF stores the verified peak at
`stage2_checkpoint/exploratory_best_test_model.pt`, its predictions and curve.
The summary ranks architectures by peak test F1, independently of dev gates.
`best_model.pt` and dev-selected metrics remain reference evidence.

These scores are **test-aware exploratory peaks**, selected on the test
set itself; they are not an independent held-out test estimate. Keep this
selection protocol explicit when reporting results. No ensemble is used.

Add `HF_TOKEN` with model-repo write access to Colab Secrets. Private GitHub
checkout also needs `GITHUB_TOKEN` with Contents: read. Select a BF16-capable
GPU. The new RUN_GROUP prevents mixing evidence with the previous dev-only
notebook. Source SHA, model/data hashes and runtime versions are frozen.
Later capacity/robustness cells define launchers and do not train on Run All.


In [ ]:
import os
import pathlib
import subprocess
import sys
import json

REPO_URL = "https://github.com/baominh5xx2/grid-repo.git"
SOURCE_REF = "main"  # First session freezes its SHA; resume reuses that SHA.
REQUIRE_GITHUB_TOKEN = False
RUN_GROUP = "vinli-arch-testpeak30-bf16-2026-10-03"
DATASETS = ["vinli"]
METHODS = ["M3_FULL", "ARCH_ALIGN256", "ARCH_REL256", "ARCH_CONDPOOL128"]
SEEDS = [42]
JOB_SPECS = [dict(dataset="vinli", experiment_id=method, seed=42) for method in METHODS]
VINLI_MAX_LENGTH = 512
EXPECTED_SPLIT_SIZES = {"train": 18282, "dev": 2255, "test": 2264}
PINNED_MODEL_NAME = "uitnlp/CafeBERT"
PINNED_MODEL_REVISION = "af76fcf2a04096b2b54b348a3e4eb48253c93c5d"
PINNED_VINLI_REVISION = "47bd78ac5d075bd00a3cb4cdd3ede4eec4acf8c2"
TRAIN_PRECISION = "bf16"
EVALUATION_PRECISION = "fp32"
USE_EMA = True
USE_MSD = True
MSD_PROBABILITIES = [0.1, 0.2, 0.3, 0.4, 0.5]
WANDB_ENABLED = False
TEST_PEAK_EXPLORATORY = True
FROZEN_FINAL = False
USE_DRIVE = True
HF_PRIVATE = False
HYPERPARAMS = dict(epochs=7, eval_steps=30, patience=0, lr=1e-5,
                  weight_decay=0.005, warmup_ratio=0.06, label_smoothing=0.02,
                  dropout=0.1, physical_batch_size=4, grad_accum=4,
                  ema_decay=0.992, ema_start_step=100)
assert DATASETS == ["vinli"] and SEEDS == [42]
assert METHODS == ["M3_FULL", "ARCH_ALIGN256", "ARCH_REL256", "ARCH_CONDPOOL128"]
assert VINLI_MAX_LENGTH == 512 and USE_EMA and USE_MSD
assert not WANDB_ENABLED and TEST_PEAK_EXPLORATORY and not FROZEN_FINAL
print(f"Configured: {len(JOB_SPECS)} ViNLI seed-42 test-aware exploratory jobs; peak test selection")


## Authenticate and freeze the checkout
Use Colab Secrets for credentials. Resume uses the original source SHA and
refuses changed configuration. Select a new RUN_GROUP for a different study.


In [ ]:
import contextlib
import tempfile
from getpass import getpass
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
from urllib.parse import urlparse

parsed_repo = urlparse(REPO_URL)
if (parsed_repo.scheme != "https" or parsed_repo.netloc != "github.com"
        or parsed_repo.query or parsed_repo.fragment):
    raise ValueError("Use a plain HTTPS github.com repository URL without credentials")
repo_parts = parsed_repo.path.removesuffix(".git").strip("/").split("/")
if len(repo_parts) != 2 or not all(repo_parts):
    raise ValueError("Expected https://github.com/OWNER/REPO.git")
GITHUB_REPOSITORY = "/".join(repo_parts)

def github_repo_info(token):
    headers = {"Accept": "application/vnd.github+json", "User-Agent": "grid-colab-bootstrap"}
    if token:
        headers["Authorization"] = "Bearer " + token
    request = Request("https://api.github.com/repos/" + GITHUB_REPOSITORY, headers=headers)
    try:
        with urlopen(request, timeout=30) as response:
            info = json.load(response)
    except HTTPError as error:
        raise RuntimeError(f"GitHub access failed (HTTP {error.code}); check token, expiry and selected repository") from None
    except URLError:
        raise RuntimeError("GitHub is unreachable; retry the access cell") from None
    if info.get("full_name", "").lower() != GITHUB_REPOSITORY.lower():
        raise RuntimeError("GitHub repository identity mismatch")
    return info

@contextlib.contextmanager
def github_git_environment(token):
    env = os.environ.copy()
    env.pop("GITHUB_TOKEN", None)
    env.update(GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="3",
               GIT_CONFIG_KEY_0="credential.helper", GIT_CONFIG_VALUE_0="",
               GIT_CONFIG_KEY_1="credential.useHttpPath", GIT_CONFIG_VALUE_1="true",
               GIT_CONFIG_KEY_2="credential.username", GIT_CONFIG_VALUE_2="x-access-token")
    with tempfile.TemporaryDirectory(prefix="grid-git-auth-") as directory:
        helper = pathlib.Path(directory) / "askpass.py"
        helper.write_text(chr(10).join([
            "#!" + sys.executable, "import os, sys",
            'print("x-access-token" if "username" in sys.argv[1].lower() else os.environ.get("GRID_GITHUB_TOKEN", ""))',
            ""]), encoding="utf-8")
        helper.chmod(0o700)
        env["GIT_ASKPASS"] = str(helper)
        if token:
            env["GRID_GITHUB_TOKEN"] = token
        else:
            env.pop("GRID_GITHUB_TOKEN", None)
        yield env

def github_git(arguments, env, cwd=None):
    result = subprocess.run(["git", *arguments], cwd=cwd, env=env,
                            text=True, capture_output=True)
    if result.returncode:
        # Avoid echoing credential-bearing remote errors or subprocess arguments.
        raise RuntimeError(f"Git failed ({arguments[0]}, exit {result.returncode}); check GITHUB_TOKEN Contents: read permission and SOURCE_REF")
    return result.stdout.strip()

_github_token = os.environ.get("GITHUB_TOKEN")
if not _github_token:
    try:
        from google.colab import userdata
        _github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        _github_token = None
if not _github_token and REQUIRE_GITHUB_TOKEN:
    _github_token = getpass("GITHUB_TOKEN (Contents: read): ").strip()
    if not _github_token:
        raise RuntimeError("GITHUB_TOKEN is required by configuration")
if _github_token:
    _github_info = github_repo_info(_github_token)
else:
    try:
        _github_info = github_repo_info(None)
    except RuntimeError as error:
        if "HTTP 404" not in str(error) and "HTTP 401" not in str(error):
            raise
        _github_token = getpass("GITHUB_TOKEN for private repository (Contents: read): ").strip()
        if not _github_token:
            raise RuntimeError("Private/unavailable repository: add GITHUB_TOKEN to Colab Secrets") from None
        _github_info = github_repo_info(_github_token)
print("GitHub repository access verified:", GITHUB_REPOSITORY,
      "| authenticated" if _github_token else "| public read")


In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
WORKSPACE = pathlib.Path("/content") if IN_COLAB else pathlib.Path.cwd() / "bf16_workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)
if IN_COLAB and USE_DRIVE:
    drive.mount("/content/drive")
    OUTPUT_ROOT = pathlib.Path("/content/drive/MyDrive/grid-repo-runs") / RUN_GROUP
else:
    OUTPUT_ROOT = WORKSPACE / "results" / RUN_GROUP
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = WORKSPACE / "grid-repo"
LOCAL_RUN_ROOT = WORKSPACE / "training" / RUN_GROUP
os.environ["HF_HOME"] = str(WORKSPACE / "hf-cache")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
previous_manifest = OUTPUT_ROOT / "batch_manifest.json"
previous = json.loads(previous_manifest.read_text()) if previous_manifest.exists() else None
target_ref = previous["git_sha"] if previous else SOURCE_REF
with github_git_environment(_github_token) as git_env:
    if not (REPO_DIR / ".git").exists():
        if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
            raise RuntimeError(f"Choose an empty checkout directory: {REPO_DIR}")
        github_git(["clone", "--no-checkout", REPO_URL, str(REPO_DIR)], git_env)
    else:
        if github_git(["remote", "get-url", "origin"], git_env, REPO_DIR) != REPO_URL:
            raise RuntimeError("Checkout origin differs; choose a fresh workspace directory")
        if github_git(["status", "--porcelain"], git_env, REPO_DIR):
            raise RuntimeError("Source checkout is dirty; preserve those edits before proceeding.")
    github_git(["fetch", "--depth", "1", "origin", target_ref], git_env, REPO_DIR)
    github_git(["checkout", "--detach", "FETCH_HEAD"], git_env, REPO_DIR)
    SOURCE_SHA = github_git(["rev-parse", "HEAD"], git_env, REPO_DIR)
    assert not github_git(["status", "--porcelain"], git_env, REPO_DIR)
del git_env, _github_token
print("Frozen source SHA:", SOURCE_SHA)
print("Persistent results:", OUTPUT_ROOT)


In [ ]:
if "_LOADED_SOURCE_SHA" in globals() and _LOADED_SOURCE_SHA != SOURCE_SHA:
    raise RuntimeError("Source SHA changed in this kernel. Restart the runtime kernel and rerun the setup cells.")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR) + "[hf]"], check=True)
sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)
_LOADED_SOURCE_SHA = SOURCE_SHA


In [ ]:
import torch
from gated_dual_ema_msd.training.precision import bf16_enabled
if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU runtime before starting this notebook.")
assert bf16_enabled(torch.device("cuda:0"))
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| Train: BF16 | Eval: FP32")


In [ ]:
from getpass import getpass
for key in ("HF_TOKEN",):
    value = os.environ.get(key)
    if not value and IN_COLAB:
        from google.colab import userdata
        try:
            value = userdata.get(key)
        except Exception:
            value = None
    if not value:
        value = getpass(f"{key}: ")
    if not value:
        raise RuntimeError(f"Missing {key}")
    os.environ[key] = value
del value
print("HF_TOKEN is present. Artifacts and metadata use HF only.")


In [ ]:
import pandas as pd
from huggingface_hub import HfApi
from gated_dual_ema_msd.cli.matrix import MatrixJob
from gated_dual_ema_msd.config.experiments import EXPERIMENTS, DATASET_MAX_LENGTHS, MODEL_NAME, MODEL_REVISION
from gated_dual_ema_msd.operations.notebook_batch import prepare_data, bind_manifest, run_batch, train_command, write_json
from gated_dual_ema_msd.training.r2_runtime import environment_metadata

if MODEL_NAME != PINNED_MODEL_NAME or MODEL_REVISION != PINNED_MODEL_REVISION:
    raise ValueError("Backbone differs from the frozen CafeBERT source")
if DATASET_MAX_LENGTHS["vinli"] != VINLI_MAX_LENGTH:
    raise ValueError("ViNLI max length must be 512")
for method in METHODS:
    experiment = EXPERIMENTS[method]
    if not experiment.use_ema or not experiment.use_msd or 42 not in experiment.seeds:
        raise ValueError(f"Architecture must preserve EMA/MSD/seed42: {method}")
JOBS = [MatrixJob(**spec) for spec in JOB_SPECS]
display(pd.DataFrame([dict(**spec, max_length=VINLI_MAX_LENGTH) for spec in JOB_SPECS]))
api = HfApi(token=os.environ["HF_TOKEN"])
HF_NAMESPACE = api.whoami()["name"]
CONFIG = {**HYPERPARAMS, "hf_namespace": HF_NAMESPACE, "hf_prefix": RUN_GROUP,
          "hf_private": HF_PRIVATE, "keep_local_checkpoints": False,
          "frozen_final": False, "test_peak_exploratory": True}
if previous and previous["config"] != CONFIG:
    raise ValueError("Settings differ from this RUN_GROUP. Choose a new RUN_GROUP.")

def verify_hf_write_access(jobs, config):
    for job in jobs:
        repo_id = (f"{config['hf_namespace']}/{config['hf_prefix']}-{job.dataset}-"
                   f"{job.experiment_id.lower().replace('_', '-')}-seed{job.seed}")
        api.create_repo(repo_id=repo_id, repo_type="model", private=config["hf_private"], exist_ok=True)
        if api.repo_info(repo_id=repo_id, repo_type="model").private != config["hf_private"]:
            raise RuntimeError(f"HF repository visibility differs: {repo_id}")
verify_hf_write_access(JOBS, CONFIG)
print("HF write access verified for", len(JOBS), "repositories; W&B disabled")


## Bind pinned train/dev/test data
Test access is explicitly enabled for this exploratory study. The preparation
flag below includes test fingerprints; training remains exploratory rather
than frozen-final. CafeBERT and all dataset revisions remain pinned.


In [ ]:
from gated_dual_ema_msd.operations.architecture_search import environment_contract
DATA_FINGERPRINTS = prepare_data(REPO_DIR, DATASETS, frozen_final=True)
if DATA_FINGERPRINTS["vinli"]["revision"] != PINNED_VINLI_REVISION:
    raise ValueError("ViNLI differs from its pinned revision")
SESSION_ENVIRONMENT = environment_metadata()
ENVIRONMENT_CONTRACT = environment_contract(SESSION_ENVIRONMENT)
BATCH_MANIFEST = {"git_sha": SOURCE_SHA, "model_name": MODEL_NAME, "model_revision": MODEL_REVISION,
                  "precision": "bf16", "evaluation_precision": "fp32", "test_locked": False,
                  "environment_contract": ENVIRONMENT_CONTRACT, "label_order": ["E", "C", "N"],
                  "seeds": SEEDS, "datasets": DATASETS, "methods": METHODS,
                  "jobs": [dict(dataset=j.dataset, experiment_id=j.experiment_id, seed=j.seed) for j in JOBS],
                  "config": CONFIG, "data_fingerprints": DATA_FINGERPRINTS,
                  "protocol": "vinli_architecture_test_peak_exploratory",
                  "prior_test_exposure": True, "selection_policy": "exploratory_test_macro_f1",
                  "dev_role": "reference only; no dev gate or early stopping",
                  "test_peak_rule": {"weight_source": "ema", "metric": "macro_f1", "ties": "earliest_step"}}
SIGNATURE = bind_manifest(OUTPUT_ROOT, BATCH_MANIFEST)
write_json(OUTPUT_ROOT / "session_environment.json", SESSION_ENVIRONMENT)
print("Manifest:", SIGNATURE)
print("Split counts:", {s: x["row_count"] for s, x in DATA_FINGERPRINTS["vinli"]["splits"].items()})


## Head preflight and four training runs
Preflight checks train/dev/test counts, max length, precision, EMA/MSD and
actual head parameters. Every child receives `--test_peak_exploratory --no_wandb`.
Interrupting the training cell stops its supervised child. Local weights are
cleaned only after immutable HF read-back verification of both checkpoints.


In [ ]:
HEAD_PREFLIGHT = []
for job in JOBS:
    completed = subprocess.run(train_command(job, CONFIG, LOCAL_RUN_ROOT) + ["--preflight"],
                               cwd=REPO_DIR, text=True, capture_output=True, check=True)
    # The backbone prints loader status before the final pretty JSON report.
    report_start = completed.stdout.rfind("\n{")
    report = json.loads(completed.stdout[report_start + 1:] if report_start >= 0 else completed.stdout)
    experiment = report["experiment"]
    msd = report["msd"]
    if (report["train_precision"] != "bf16" or report["eval_precision"] != "fp32"
            or report["max_length"] != 512 or report["split_sizes"] != EXPECTED_SPLIT_SIZES
            or experiment["experiment_id"] != job.experiment_id
            or experiment["use_ema"] is not True or experiment["use_msd"] is not True
            or msd["msd_enabled"] is not True or msd["msd_num_paths"] != 5
            or msd["msd_shared_classifier"] is not True
            or msd["msd_dropout_probabilities"] != MSD_PROBABILITIES
            or report["parameters"]["additional_trainable_head_parameters"] <= 0):
        raise ValueError(f"Architecture preflight protocol mismatch: {job.experiment_id}")
    HEAD_PREFLIGHT.append(report)
write_json(OUTPUT_ROOT / "head_preflight.json", {"jobs": HEAD_PREFLIGHT})
display(pd.DataFrame([dict(experiment_id=r["experiment"]["experiment_id"],
                           head_parameters=r["parameters"]["additional_trainable_head_parameters"],
                           **r["split_sizes"]) for r in HEAD_PREFLIGHT]))


In [ ]:
try:
    run_batch(JOBS, REPO_DIR, LOCAL_RUN_ROOT, OUTPUT_ROOT, CONFIG, BATCH_MANIFEST)
except KeyboardInterrupt:
    write_json(OUTPUT_ROOT / "progress.json", {"state": "interrupted", "resume": "rerun setup and batch cells"})
    raise
except Exception as error:
    write_json(OUTPUT_ROOT / "progress.json", {"state": "failed", "error_type": type(error).__name__,
                                              "resume": "fix the logged failure, then rerun the batch cell"})
    raise


## Highest observed test checkpoint per architecture
The table below uses peak-test metrics and checkpoint paths. It does not
substitute the final test score of the dev-selected checkpoint. All four
verified pilot runs are required before choosing the seed-42 winner.


In [ ]:
from gated_dual_ema_msd.operations.architecture_test_peak import test_peak_table, select_peak_candidate
TEST_PEAK_TABLE = test_peak_table(OUTPUT_ROOT)
display(TEST_PEAK_TABLE)
TEST_PEAK_TABLE.to_csv(OUTPUT_ROOT / "exploratory_architecture_test_peaks.csv", index=False)
try:
    SELECTED_CANDIDATE = select_peak_candidate(OUTPUT_ROOT)
except ValueError as error:
    SELECTED_CANDIDATE = None
    print("Pilot selection pending:", error)
else:
    print("TEST-AWARE EXPLORATORY winner:", SELECTED_CANDIDATE)
print("Verified peak weights, predictions and curves:", OUTPUT_ROOT)


## Optional capacity control
Invoke `launch_capacity_control(authorize=True)` explicitly after inspecting
the peak results. It runs one seed-42 parameter control for the selected new
architecture. If M3 wins, there is no additional architecture control.


In [ ]:
import copy
from gated_dual_ema_msd.operations.architecture_test_peak import peak_capacity_control_jobs, select_peak_candidate
from gated_dual_ema_msd.operations.architecture_search import environment_contract

def launch_capacity_control(*, authorize=False):
    if authorize is not True:
        raise ValueError("Invoke with authorize=True after reviewing test-aware pilot evidence")
    if environment_contract(environment_metadata()) != BATCH_MANIFEST["environment_contract"]:
        raise ValueError("Current runtime differs from the frozen study")
    selected = select_peak_candidate(OUTPUT_ROOT)
    jobs = peak_capacity_control_jobs(OUTPUT_ROOT)
    if not jobs:
        print("M3 control is the winner; no new architecture capacity control")
        return None
    group = RUN_GROUP + "-capacity"
    output_root = OUTPUT_ROOT.parent / group
    config = {**CONFIG, "hf_prefix": group}
    manifest = copy.deepcopy(BATCH_MANIFEST)
    manifest.update(protocol="vinli_architecture_test_peak_capacity", methods=[jobs[0].experiment_id],
                    seeds=[42], jobs=[dict(dataset=j.dataset, experiment_id=j.experiment_id, seed=j.seed) for j in jobs],
                    config=config, screening_manifest_sha256=SIGNATURE, selected_candidate=selected)
    bind_manifest(output_root, manifest)
    verify_hf_write_access(jobs, config)
    run_batch(jobs, REPO_DIR, LOCAL_RUN_ROOT.parent / group, output_root, config, manifest)
    return output_root


## Multi-seed test-peak robustness
Explicitly invoke `launch_confirmation("<winner>", authorize=True)`.
This runs only M3 and the test-peak winner on seeds 2024/3407, with the same
full budget and test scanning. If M3 wins, run its two extra seeds only.
All seed peaks remain reported; this repetition does not undo test selection.


In [ ]:
import copy
from gated_dual_ema_msd.operations.architecture_test_peak import peak_confirmation_jobs

def launch_confirmation(selected_candidate, *, authorize=False):
    if authorize is not True:
        raise ValueError("Invoke with authorize=True after reviewing the test-aware pilot")
    if environment_contract(environment_metadata()) != BATCH_MANIFEST["environment_contract"]:
        raise ValueError("Current runtime differs from the frozen study")
    jobs = peak_confirmation_jobs(OUTPUT_ROOT, selected_candidate)
    group = RUN_GROUP + "-confirm"
    output_root = OUTPUT_ROOT.parent / group
    config = {**CONFIG, "hf_prefix": group}
    manifest = copy.deepcopy(BATCH_MANIFEST)
    manifest.update(protocol="vinli_architecture_test_peak_robustness", seeds=[2024, 3407],
                    methods=list(dict.fromkeys(j.experiment_id for j in jobs)), config=config,
                    jobs=[dict(dataset=j.dataset, experiment_id=j.experiment_id, seed=j.seed) for j in jobs],
                    screening_manifest_sha256=SIGNATURE, selected_candidate=selected_candidate)
    bind_manifest(output_root, manifest)
    verify_hf_write_access(jobs, config)
    run_batch(jobs, REPO_DIR, LOCAL_RUN_ROOT.parent / group, output_root, config, manifest)
    return output_root


## Report existing peaks without more inference
After the extra seeds finish, invoke `show_robustness("<winner>")` to export
all seed peaks and their mean/std. It only reads verified artifacts; it
performs no additional test inference or checkpoint ensemble.


In [ ]:
from gated_dual_ema_msd.operations.architecture_test_peak import peak_robustness_summary

def show_robustness(candidate):
    root = OUTPUT_ROOT.parent / (RUN_GROUP + "-confirm")
    runs, summary = peak_robustness_summary(OUTPUT_ROOT, root, candidate)
    runs.to_csv(root / "exploratory_architecture_peak_runs.csv", index=False)
    summary.to_csv(root / "exploratory_architecture_peak_summary.csv", index=False)
    display(runs)
    display(summary)
    return runs, summary
